## Clean Up COCO JSON

Remove segmentation, and overwrite iscrowd.

BePLi dataset supports segmentation, but we are not going to be doing that.
We want to force sahi image slicer to use bbox.

The cleaned up json will have `clean_*` prefix.

In [2]:
import json
import os

# --- CONFIGURATION ---
# Define the path to your annotations
base_dir = "plastic_coco/annotation"
files_to_fix = ["train.json", "val.json", "test.json"]

def sanitize_json(file_name):
    path = os.path.join(base_dir, file_name)
    if not os.path.exists(path):
        print(f"Skipping {file_name}, file not found.")
        return

    print(f"Sanitizing {file_name}...")
    with open(path, 'r') as f:
        data = json.load(f)

    count_fixed = 0
    for ann in data['annotations']:
        # 1. Remove segmentation (forces SAHI to use bbox)
        if 'segmentation' in ann:
            del ann['segmentation']
            count_fixed += 1
        
        # 2. Fix 'iscrowd' (RLE is often used for 'iscrowd=1' objects)
        # We set it to 0 so the model treats them as valid targets
        if 'iscrowd' in ann:
            ann['iscrowd'] = 0

    # Save as a new file so we don't overwrite the original
    new_path = os.path.join(base_dir, f"clean_{file_name}")
    with open(new_path, 'w') as f:
        json.dump(data, f)
    
    print(f"Fixed {count_fixed} annotations. Saved to: {new_path}")

# --- EXECUTE ---
for f in files_to_fix:
    sanitize_json(f)

Sanitizing train.json...
Fixed 73895 annotations. Saved to: plastic_coco/annotation/clean_train.json
Sanitizing val.json...
Fixed 23015 annotations. Saved to: plastic_coco/annotation/clean_val.json
Sanitizing test.json...
Fixed 22282 annotations. Saved to: plastic_coco/annotation/clean_test.json


## Data Slicing

This is done because BePLi dataset doesn't have uniform resolution. A larger imgsz will support for high resolution images but cause low resolution images to be upscaled. A smaller imgsz will support for low resolution images but cause high resolution images to lose quality, small details would then become few pixels.

After slicing we convert COCO json to YOLO .txt files for labels. After that we also copy the test dataset so that it can be used in 1 portable dataset. This also makes the dataset yml file cleaner. 

The output is in `plastic_sliced_dataset` folder.


In [4]:
import json
import os
import shutil
from pathlib import Path

from sahi.slicing import slice_coco

# --- CONFIGURATION ---
BASE_DIR = Path("plastic_coco")
IMG_DIR = BASE_DIR / "images"
ANN_DIR = BASE_DIR / "annotation"

# Output dataset in standard YOLO layout
OUTPUT_BASE = Path("plastic_sliced_dataset")
OUTPUT_IMAGES = OUTPUT_BASE / "images"
OUTPUT_LABELS = OUTPUT_BASE / "labels"

def _load_categories_for_class_map() -> tuple[dict[int, int], list[str]]:
    """Build a stable COCO category-id -> contiguous 0..N-1 mapping."""
    preferred = [
        ANN_DIR / "all_plastic_coco.json",
        ANN_DIR / "clean_train.json",
        ANN_DIR / "train.json",
        ANN_DIR / "clean_val.json",
        ANN_DIR / "val.json",
        ANN_DIR / "clean_test.json",
        ANN_DIR / "test.json",
    ]
    for json_path in preferred:
        if json_path.exists():
            data = json.loads(json_path.read_text())
            cats = data.get("categories", [])
            if cats:
                cats_sorted = sorted(cats, key=lambda c: c["id"])
                cat_id_map = {cat["id"]: i for i, cat in enumerate(cats_sorted)}
                class_names = [cat["name"] for cat in cats_sorted]
                return cat_id_map, class_names
    raise FileNotFoundError(f"No COCO categories found under {ANN_DIR}")

CAT_ID_MAP, CLASS_NAMES = _load_categories_for_class_map()

def _coco_bbox_to_yolo_xywhn(bbox, img_w, img_h):
    # COCO bbox: [x_min, y_min, width, height]
    x, y, w, h = bbox
    img_w = float(img_w)
    img_h = float(img_h)

    # Clip to image bounds
    x1 = max(0.0, float(x))
    y1 = max(0.0, float(y))
    x2 = min(img_w, float(x) + float(w))
    y2 = min(img_h, float(y) + float(h))
    w = x2 - x1
    h = y2 - y1
    if w <= 0 or h <= 0:
        return None

    x_center = (x1 + w / 2) / img_w
    y_center = (y1 + h / 2) / img_h
    w_norm = w / img_w
    h_norm = h / img_h
    return x_center, y_center, w_norm, h_norm

def _convert_coco_json_to_yolo_labels(coco_json_path: Path, labels_out_dir: Path):
    """Convert a COCO json into YOLO txt labels."""
    print(f"Reading JSON for conversion: {coco_json_path}")
    data = json.loads(coco_json_path.read_text())
    images = {img["id"]: img for img in data.get("images", [])}

    labels_out_dir.mkdir(parents=True, exist_ok=True)

    labels_by_stem: dict[str, list[str]] = {}
    skipped = 0

    for ann in data.get("annotations", []):
        img_info = images.get(ann.get("image_id"))
        if not img_info:
            skipped += 1
            continue

        file_stem = Path(img_info.get("file_name", "")).stem
        img_w = img_info.get("width")
        img_h = img_info.get("height")
        if not file_stem or not img_w or not img_h:
            skipped += 1
            continue

        class_id = CAT_ID_MAP.get(ann.get("category_id"))
        if class_id is None:
            skipped += 1
            continue

        yolo = _coco_bbox_to_yolo_xywhn(ann.get("bbox"), img_w, img_h)
        if yolo is None:
            skipped += 1
            continue

        x, y, w, h = yolo
        labels_by_stem.setdefault(file_stem, []).append(
            f"{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n"
        )

    # Overwrite labels for idempotent reruns
    for stem, lines in labels_by_stem.items():
        (labels_out_dir / f"{stem}.txt").write_text("".join(lines))

    print(f"Converted -> {len(labels_by_stem)} label files (skipped {skipped} anns)")

def process_split(split_name: str, do_slice: bool):
    print(f"\n--- Processing {split_name} set (Slicing: {do_slice}) ---")

    image_dir = IMG_DIR / split_name
    input_json = ANN_DIR / f"clean_{split_name}.json"
    if not input_json.exists():
        input_json = ANN_DIR / f"{split_name}.json"
    
    # Allow missing test JSON if you just have images, though usually we have a json
    if not input_json.exists():
        print(f"Warning: No JSON found for {split_name} at {input_json}. Skipping conversion.")
        return

    out_images_dir = OUTPUT_IMAGES / split_name
    out_labels_dir = OUTPUT_LABELS / split_name
    out_images_dir.mkdir(parents=True, exist_ok=True)
    out_labels_dir.mkdir(parents=True, exist_ok=True)

    target_json_path = input_json

    if do_slice:
        # 1) SLICE (creates sliced images + sliced COCO json)
        print(f"Slicing images from: {image_dir}")
        _, sliced_coco_path = slice_coco(
            coco_annotation_file_path=str(input_json),
            image_dir=str(image_dir),
            output_coco_annotation_file_name=f"sliced_{split_name}.json",
            output_dir=str(out_images_dir),
            slice_height=640,
            slice_width=640,
            overlap_height_ratio=0.2,
            overlap_width_ratio=0.2,
            min_area_ratio=0.1,
            verbose=False,
        )
        target_json_path = Path(sliced_coco_path)
        print(f"Created sliced COCO json: {target_json_path}")

    else:
        # 2) NO SLICE (Copy images + use original JSON)
        print(f"Copying images (no slicing) from: {image_dir}")
        valid_extensions = {".jpg", ".jpeg", ".png", ".bmp"}
        count = 0
        if image_dir.exists():
            for f in image_dir.iterdir():
                if f.suffix.lower() in valid_extensions:
                    shutil.copy2(f, out_images_dir / f.name)
                    count += 1
        print(f"Copied {count} images to {out_images_dir}")
        # target_json_path remains the original input_json

    # 3) CONVERT (COCO json -> YOLO labels)
    print(f"Converting annotations to YOLO labels: {out_labels_dir}")
    _convert_coco_json_to_yolo_labels(target_json_path, out_labels_dir)

    print(f"Finished {split_name}.")

# --- EXECUTION ---

# Slice Train and Val
process_split("train", do_slice=True)
process_split("val", do_slice=True)

# Copy Test (Do NOT Slice)
process_split("test", do_slice=False)

print("\n--- Done! ---")
print(f"Dataset ready at: {OUTPUT_BASE}")
print(f"Classes (YOLO order): {CLASS_NAMES}")


--- Processing train set (Slicing: True) ---
Slicing images from: plastic_coco/images/train


100%|██████████| 2226/2226 [02:54<00:00, 12.73it/s]


Created sliced COCO json: plastic_sliced_dataset/images/train/sliced_train.json_coco.json
Converting annotations to YOLO labels: plastic_sliced_dataset/labels/train
Reading JSON for conversion: plastic_sliced_dataset/images/train/sliced_train.json_coco.json
Converted -> 6233 label files (skipped 0 anns)
Finished train.

--- Processing val set (Slicing: True) ---
Slicing images from: plastic_coco/images/val


100%|██████████| 742/742 [00:56<00:00, 13.08it/s]


Created sliced COCO json: plastic_sliced_dataset/images/val/sliced_val.json_coco.json
Converting annotations to YOLO labels: plastic_sliced_dataset/labels/val
Reading JSON for conversion: plastic_sliced_dataset/images/val/sliced_val.json_coco.json
Converted -> 2024 label files (skipped 0 anns)
Finished val.

--- Processing test set (Slicing: False) ---
Copying images (no slicing) from: plastic_coco/images/test
Copied 741 images to plastic_sliced_dataset/images/test
Converting annotations to YOLO labels: plastic_sliced_dataset/labels/test
Reading JSON for conversion: plastic_coco/annotation/clean_test.json
Converted -> 741 label files (skipped 0 anns)
Finished test.

--- Done! ---
Dataset ready at: plastic_sliced_dataset
Classes (YOLO order): ['plastic_litter']


## Train

In [5]:
from ultralytics import YOLO

model = YOLO('yolo11m.pt')  # load a pretrained model (nano version)

# Train the model
results = model.train(
    data='bepliv1_sliced.yml', 
    epochs=1,
    patience=20,
    imgsz=640,
    batch=1,
    project='plastic_project', # Name of the folder where results will be saved
    # device=[0, 1],
)

New https://pypi.org/project/ultralytics/8.3.244 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+xpu CPU (Intel Core Ultra 7 258V)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=bepliv1_sliced.yml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=Fal

KeyboardInterrupt: 

## Test

In [ ]:
from pathlib import Path
from PIL import Image

best_weights = Path('plastic_project/train3/weights/best.pt')

if not best_weights.exists():
    raise FileNotFoundError(f"{best_weights} not found; rerun training or adjust the path.")

imgs = ['beach1.jpg', 'beach2.jpg', 'beach3.jpg']

for img in imgs:
    test_image = Path(img)
    if not test_image.exists():
        raise FileNotFoundError(f"{test_image} not found; place the sample image alongside the notebook.")
    
    infer_model = YOLO(str(best_weights))
    results = infer_model(test_image, conf=0.5)  # predict on an image
    
    # print(result)
    # Access the results
    for result in results:
        im_bgr = result.plot()
        xywh = result.boxes.xywh  # center-x, center-y, width, height
        xywhn = result.boxes.xywhn  # normalized
        xyxy = result.boxes.xyxy  # top-left-x, top-left-y, bottom-right-x, bottom-right-y
        xyxyn = result.boxes.xyxyn  # normalized
        names = [result.names[cls.item()] for cls in result.boxes.cls.int()]  # class name of each box
        confs = result.boxes.conf  # confidence score of each box
    
        print(names, confs)
    
        # Display the test image that was classified
        from IPython.display import display
        display(Image.fromarray(im_bgr))